
# BTVN: Tìm hiểu các mô hình CNN điển hình — VGG, ResNet, EfficientNet, YOLO

**Tác giả:** ChatGPT • **Ngày tạo:** 2025-10-16 16:06  
**Ngôn ngữ:** Python (PyTorch)

1. **VGG** — Phân loại ảnh (kiến trúc conv 3×3 xếp chồng).  
2. **ResNet** — Phân loại ảnh (residual connections).  
3. **EfficientNet** — Phân loại ảnh (compound scaling).  
4. **YOLO** — Phát hiện đối tượng (one‑stage).

> Lần đầu chạy có thể cần Internet để tải trọng số pretrain.


## 0) Cài đặt & import

In [ ]:

# (Tuỳ chọn) Cài đặt khi thiếu
# !pip install -U torch torchvision torchaudio
# !pip install -U ultralytics
# !pip install -U matplotlib pillow requests

import os, io, sys, platform, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests

import torch, torchvision
from torchvision import transforms

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Platform:", platform.platform())

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device)


## 1) Tiện ích dùng chung

In [ ]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def load_image(path_or_url, to_rgb=True):
    try:
        if str(path_or_url).startswith(("http://","https://")):
            resp = requests.get(path_or_url, timeout=15)
            resp.raise_for_status()
            img = Image.open(io.BytesIO(resp.content))
        else:
            img = Image.open(path_or_url)
    except Exception as e:
        print("Không thể tải ảnh:", e)
        raise
    if to_rgb and img.mode != "RGB":
        img = img.convert("RGB")
    return img

def show_image(img, title=None):
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    if title: plt.title(title)
    plt.show()

def preprocess_imagenet(img_pil, size=224):
    tfm = transforms.Compose([
        transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return tfm(img_pil).unsqueeze(0)


### Ảnh ví dụ

In [ ]:

IMG_URL = "https://images.unsplash.com/photo-1568572933382-74d440642117?q=80&w=640"
IMG_PATH = None  # ví dụ: r"D:\pictures\my_dog.jpg"

img_source = IMG_PATH if IMG_PATH else IMG_URL
img = load_image(img_source, to_rgb=True)
show_image(img, "Ảnh đầu vào (demo)")



---
# 2) VGG (VGG16) — Phân loại ảnh


In [ ]:

from torchvision.models import vgg16, VGG16_Weights

try:
    weights = VGG16_Weights.DEFAULT
    categories = weights.meta.get("categories", None)
    model_vgg = vgg16(weights=weights).to(device).eval()
    print("Đã tải VGG16 pretrained.")
except Exception as e:
    print("Không tải được trọng số VGG16, dùng random init:", e)
    model_vgg = vgg16(weights=None).to(device).eval()
    categories = None

x = preprocess_imagenet(img, size=224).to(device)
with torch.no_grad():
    logits = model_vgg(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

top5_idx = np.argsort(-probs)[:5]
print("Top-5 (VGG16):")
for i in top5_idx:
    label = categories[i] if categories else f"class_{i}"
    print(f"  {label:25s} prob={probs[i]:.4f}")



---
# 3) ResNet (ResNet18) — Phân loại ảnh


In [ ]:

from torchvision.models import resnet18, ResNet18_Weights

try:
    weights = ResNet18_Weights.DEFAULT
    categories = weights.meta.get("categories", None)
    model_res = resnet18(weights=weights).to(device).eval()
    print("Đã tải ResNet18 pretrained.")
except Exception as e:
    print("Không tải được trọng số ResNet18, dùng random init:", e)
    model_res = resnet18(weights=None).to(device).eval()
    categories = None

x = preprocess_imagenet(img, size=224).to(device)
with torch.no_grad():
    logits = model_res(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

top5_idx = np.argsort(-probs)[:5]
print("Top-5 (ResNet18):")
for i in top5_idx:
    label = categories[i] if categories else f"class_{i}"
    print(f"  {label:25s} prob={probs[i]:.4f}")



---
# 4) EfficientNet (B0) — Phân loại ảnh


In [ ]:

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

try:
    weights = EfficientNet_B0_Weights.DEFAULT
    categories = weights.meta.get("categories", None)
    model_eff = efficientnet_b0(weights=weights).to(device).eval()
    print("Đã tải EfficientNet-B0 pretrained.")
except Exception as e:
    print("Không tải được trọng số EfficientNet-B0, dùng random init:", e)
    model_eff = efficientnet_b0(weights=None).to(device).eval()
    categories = None

x = preprocess_imagenet(img, size=224).to(device)
with torch.no_grad():
    logits = model_eff(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

top5_idx = np.argsort(-probs)[:5]
print("Top-5 (EfficientNet-B0):")
for i in top5_idx:
    label = categories[i] if categories else f"class_{i}"
    print(f"  {label:25s} prob={probs[i]:.4f}")



---
# 5) YOLO (Ultralytics YOLOv8) — Phát hiện đối tượng


In [ ]:

try:
    from ultralytics import YOLO
    have_ultralytics = True
except Exception as e:
    have_ultralytics = False
    print("Chưa có 'ultralytics'. Cài bằng: pip install -U ultralytics")

if have_ultralytics:
    try:
        yolo = YOLO('yolov8n.pt')  # tải lần đầu nếu chưa có
        results = yolo.predict(img, conf=0.25, verbose=False)
        res = results[0]
        annotated = res.plot()[:, :, ::-1]  # BGR->RGB
        plt.figure(figsize=(6,6))
        plt.imshow(annotated)
        plt.axis("off")
        plt.title("YOLOv8 — Kết quả phát hiện")
        plt.show()

        if res.boxes is not None and len(res.boxes) > 0:
            print("Các đối tượng phát hiện:")
            for b in res.boxes:
                cls_id = int(b.cls.item())
                conf  = float(b.conf.item())
                name  = res.names.get(cls_id, str(cls_id))
                print(f"  - {name}: conf={conf:.2f}")
        else:
            print("Không phát hiện được đối tượng nào với ngưỡng conf hiện tại.")
    except Exception as e:
        print("Không thể chạy YOLO:", e)
        print("Bạn có thể tải sẵn yolov8n.pt và thay đường dẫn local trong YOLO('path/to/yolov8n.pt').")



---
## 6) Ứng dụng thực tế (gợi ý nhanh)

- **VGG/ResNet/EfficientNet (Classification):**
  - Phân loại sản phẩm thương mại điện tử, bệnh lá cây, nhận diện món ăn, chất lượng bề mặt.
  - **Transfer Learning**: thay head cuối và fine‑tune với dữ liệu riêng.

- **YOLO (Detection):**
  - Giám sát giao thông/nhà xưởng, đếm người/xe, phát hiện nón bảo hiểm/khẩu trang, phát hiện logo, kiểm tra đóng gói.

### Bài mở rộng
- Thử thay ảnh đầu vào bằng ảnh nghiệp vụ thật sự của bạn.  
- Với YOLO, thử trên **video** hoặc **webcam** bằng `yolo.predict(source=0)`.
